### Imports and Configuration


In [ ]:
# Import shared libraries, metrics, and utilities used across the full CICIoT2023 modeling workflow.
# Keep metric imports centralized so split evaluation and cross-validation use consistent definitions.
# Configure warnings once to keep lengthy experiment logs readable during repeated notebook runs.

import numpy as np
import pandas as pd
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, fbeta_score)
from sklearn.model_selection import train_test_split

### Paths and Data Loading


In [ ]:
# Build project-relative paths to keep the notebook portable across machines and environments.
# Load prepared CICIoT2023 train/validation/test splits so each algorithm is tested on identical partitions.
# Ensure output directories exist before fitting so artifact persistence does not fail at completion.

from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')


NOTEBOOK_DIR =  Path('/content/drive/MyDrive/MLmodeling/XAI/notebooks_v1')
BASE_DIR     = Path('/content/drive/MyDrive/MLmodeling/XAI')
DATASET_DIR  = BASE_DIR / 'Datasets' / 'CICIOT23'
SPLITS_DIR   = BASE_DIR / 'splits' / 'CICIoT2023'
MODEL_DIR    = BASE_DIR / 'models' / 'CICIoT2023'
RESULT_DIR   = BASE_DIR / 'results' / 'CICIoT2023'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print('Loading splits...')

# Val and Test are small   load fully
X_val  = pd.read_csv(SPLITS_DIR / 'X_val.csv',  dtype=np.float32).values
X_test = pd.read_csv(SPLITS_DIR / 'X_test.csv', dtype=np.float32).values
y_val  = pd.read_csv(SPLITS_DIR / 'y_val.csv').squeeze().astype(np.int8).values
y_test = pd.read_csv(SPLITS_DIR / 'y_test.csv').squeeze().astype(np.int8).values

# Train is large   load in chunks and stack
print('Loading X_train in chunks (large file)...')
chunks = []
for chunk in pd.read_csv(SPLITS_DIR / 'X_train.csv', dtype=np.float32, chunksize=200_000):
    chunks.append(chunk.values)
X_train = np.vstack(chunks)
del chunks

y_train = pd.read_csv(SPLITS_DIR / 'y_train.csv').squeeze().astype(np.int8).values

print(f'Train : {X_train.shape} | Attack ratio: {y_train.mean():.2%}')
print(f'Val   : {X_val.shape}   | Attack ratio: {y_val.mean():.2%}')
print(f'Test  : {X_test.shape}  | Attack ratio: {y_test.mean():.2%}')

Mounted at /content/drive
Loading splits...
Loading X_train in chunks (large file)...
Train : (4152260, 40) | Attack ratio: 96.95%
Val   : (1176851, 40)   | Attack ratio: 97.66%
Test  : (1176851, 40)  | Attack ratio: 97.65%


### Model Definition


In [ ]:
# Define LOF with explicit hyperparameters so this run stays reproducible and easy to compare.
# Keep model initialization separate from training/evaluation so tuning edits remain localized.
# `novelty=True` enables scoring on unseen validation/test examples after training.

ARTIFACT_NAME = "CIC_16_LOF"
MODEL_NAME = "LOF"
# CICIoT2023: 97% attack, 3% normal -> normal is the rare class.
# LOF treats the majority as inliers (attack) and minority as outliers (normal).
# So: -1 (outlier) = normal (0),  +1 (inlier) = attack (1)    predict_binary below.
# contamination = fraction of normal in train = ~3%.
# algorithm='ball_tree' essential   default brute-force is O(n^2) and hangs on 400k+ rows.
model = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.03,      # ~3% normal ratio in CICIoT2023 train
    novelty=True,
    algorithm='ball_tree',
    leaf_size=40,
    n_jobs=-1
)


### Train and Evaluate


In [ ]:
# Fit the model and evaluate train/validation/test in one pass to expose overfitting or underfitting quickly.
# Report threshold metrics (accuracy, precision, recall, F1, F2) and ranking metrics (ROC-AUC, PR-AUC).
# PR-AUC is particularly informative for imbalanced attack detection, so it complements ROC-AUC here.

from sklearn.metrics import roc_auc_score, average_precision_score


from sklearn.model_selection import train_test_split as tts
X_fit, _, y_fit, _ = train_test_split(
    X_train, y_train,
    train_size=0.70,
    random_state=42,
    stratify=y_train
)

model.fit(X_fit)
print('Fitting complete.')

def predict_binary(model, X):
    # CICIoT: attack=majority=inlier(+1), normal=minority=outlier(-1)
    # -1 -> 0 (normal),  +1 -> 1 (attack)
    return np.where(model.predict(X) == -1, 0, 1)

def predict_scores(model, X):
    # decision_function: more negative = more anomalous (more likely normal here)
    # negate so higher score = more likely attack
    return -model.decision_function(X)

# Evaluate on sample train (not full 4M   too slow for LOF predict)
print('Predicting on train sample...')
y_pred_train  = predict_binary(model, X_fit)
scores_train  = predict_scores(model, X_fit)

print('Predicting on val...')
y_pred_val    = predict_binary(model, X_val)
scores_val    = predict_scores(model, X_val)

print('Predicting on test...')
y_pred_test   = predict_binary(model, X_test)
scores_test   = predict_scores(model, X_test)

def eval_lof(y_true, y_pred, scores, split_name):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    f2   = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    roc  = roc_auc_score(y_true, scores)
    pr   = average_precision_score(y_true, scores)
    print(f'{split_name:6s}   Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} '
          f'| F1: {f1:.4f} | F2: {f2:.4f} | ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f}')
    return dict(accuracy=acc, precision=prec, recall=rec,
                f1=f1, f2=f2, roc_auc=roc, pr_auc=pr)

train_metrics = eval_lof(y_fit,  y_pred_train, scores_train, 'Train')
val_metrics   = eval_lof(y_val,  y_pred_val,   scores_val,   'Val')
test_metrics  = eval_lof(y_test, y_pred_test,  scores_test,  'Test')

train_test_gap = round(train_metrics['f1'] - test_metrics['f1'], 6)


### Model Evaluation (Unsupervised - No cross validation)

In [ ]:
# LOF is an unsupervised learning model.
# It does not use labeled data (y), so techniques like Stratified K-Fold CV
# cannot be applied because they rely on class labels to split data.

# In unsupervised learning:
# - There is no ground truth for validation during training
# - Metrics like F1, ROC require true labels, which are not available
# - Hence, traditional cross-validation is not applicable

# Instead, we evaluate stability using:
# - Train / Validation / Test consistency
# - Distribution of anomaly scores
# - Domain-based validation (if labels available externally)


cv_results = {
    "cv_train_f1_mean": None,
    "cv_val_f1_mean":   None,
    "cv_val_f1_std":    None,
    "cv_val_roc_mean":  None,
    "cv_val_pr_mean":   None,
    "cv_gap":           None,
}
train_test_gap = None
print("CV skipped   unsupervised model")
print("Stability assessed via train / val / test consistency")

### Save Results and Model Artifact


In [ ]:
# Assemble a standardized result row so this notebook remains compatible with the shared benchmark schema.
# Save this notebook's latest metrics snapshot to CSV (default `to_csv` overwrites on rerun).
# Persist the trained model artifact for downstream interpretability workflows such as SHAP and LIME.

row = {
    'dataset':         'CICIoT2023',
    'model':           MODEL_NAME,
    'split':           '70/15/15',
    'test_accuracy':   round(test_metrics['accuracy'],  6),
    'test_precision':  round(test_metrics['precision'], 6),
    'test_recall':     round(test_metrics['recall'],    6),
    'test_f1':         round(test_metrics['f1'],        6),
    'test_f2':         round(test_metrics['f2'],        6),
    'test_roc_auc':    round(test_metrics['roc_auc'],   6),
    'test_pr_auc':     round(test_metrics['pr_auc'],    6),
    'val_f1':          round(val_metrics['f1'],         6),
    'val_f2':          round(val_metrics['f2'],         6),
    'val_roc_auc':     round(val_metrics['roc_auc'],    6),
    'val_pr_auc':      round(val_metrics['pr_auc'],     6),
    'train_f1':        round(train_metrics['f1'],       6),
    'cv_val_f1_mean':  None,
    'cv_val_f1_std':   None,
    'cv_val_roc_mean': None,
    'cv_val_pr_mean':  None,
    'cv_gap':          None,
    'train_test_gap':  train_test_gap,
}

pd.DataFrame([row]).to_csv(RESULT_DIR / f'{ARTIFACT_NAME}.csv', index=False)
joblib.dump(model, MODEL_DIR / f'{ARTIFACT_NAME}.pkl')

print(f'{ARTIFACT_NAME}   saved.')
print(pd.DataFrame([row]).T.to_string(header=False))


In [ ]:
# Widget inference setup: load model + feature defaults + optional preprocessing artifacts.
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Resolve project root robustly so this cell can run standalone.
# Previous code for BASE_DIR resolution was removed to align with earlier explicitly defined BASE_DIR.
BASE_DIR = Path('/content/drive/MyDrive/MLmodeling/XAI')

FEATURES_FILE = BASE_DIR / 'features' / 'CICIoT2023_features.json'
MODEL_FILE = BASE_DIR / 'models' / 'CICIoT2023' / 'CIC_16_LOF.pkl'
SPLIT_DIR = BASE_DIR / 'splits' / 'CICIoT2023'

assert FEATURES_FILE.exists(), f'Feature defaults not found: {FEATURES_FILE}'
assert MODEL_FILE.exists(), f'Model file not found: {MODEL_FILE}'

with open(FEATURES_FILE, 'r', encoding='utf-8') as f:
    CIC_DEFAULTS = json.load(f)

# Set CAT_OPTIONS to an empty dictionary as per user request to skip categorical features.
CAT_OPTIONS = {}

# Exclude non-predictive raw columns if they still exist in feature JSON.
EXCLUDE_COLS = {'label'}
CIC_DEFAULTS = {k: v for k, v in CIC_DEFAULTS.items() if k not in EXCLUDE_COLS}

INFER_MODEL = joblib.load(MODEL_FILE)

INFER_PREPROCESSOR = None
for pp_name in ('preprocessor.joblib', 'preprocessor.pkl'):
    pp_path = SPLIT_DIR / pp_name
    if pp_path.exists():
        INFER_PREPROCESSOR = joblib.load(pp_path)
        break

INFER_FEATURE_NAMES = None
fn_path = SPLIT_DIR / 'feature_names.csv'
if fn_path.exists():
    INFER_FEATURE_NAMES = pd.read_csv(fn_path, header=None).squeeze().tolist()

print(f'Loaded model: {MODEL_FILE.name}')
print(f'Feature defaults: {len(CIC_DEFAULTS)}')
print(f'Preprocessor loaded: {INFER_PREPROCESSOR is not None}')


In [ ]:
if 'CIC_DEFAULTS' not in globals():
    print('Run the previous widget setup cell first.')
else:
    input_widgets = {}

    # Removed the hardcoded CAT_OPTIONS as per user request to skip categorical features.
    # The widget will now only use numerical inputs based on CIC_DEFAULTS.

    def make_widget(col, default):
        # Now assuming all inputs should be numerical based on user request.
        # Handle int and float specifically, otherwise default to float.
        if isinstance(default, int) and not isinstance(default, bool):
            return widgets.IntText(value=int(default), layout=widgets.Layout(width='360px', min_width='300px', height='34px'))
        # All other cases, including floats and any string defaults (which will be attempted to convert to float)
        # This will raise a ValueError if a string cannot be converted to float, but per user request, we are skipping
        # explicit categorical handling, implying these should be numerical for the model.
        return widgets.FloatText(value=float(default), step=0.01, layout=widgets.Layout(width='360px', min_width='300px', height='34px'))

    def to_group_name(col_name):
        if '.' in col_name:
            return col_name.split('.', 1)[0]
        if '_' in col_name:
            return col_name.split('_', 1)[0]
        return 'other'

    feature_groups = {}
    for col, default in CIC_DEFAULTS.items():
        w = make_widget(col, default)
        input_widgets[col] = w
        group = to_group_name(col)
        feature_groups.setdefault(group, []).append(col)

    sorted_groups = sorted(feature_groups.keys())
    group_boxes = []

    for group in sorted_groups:
        rows = []
        for col in feature_groups[group]:
            label = widgets.HTML(
                value=f'<div style="font-family:monospace; font-size:13px;">{col}</div>',
                layout=widgets.Layout(width='430px', min_width='360px')
            )
            rows.append(
                widgets.HBox(
                    [label, input_widgets[col]],
                    layout=widgets.Layout(justify_content='space-between', width='100%')
                )
            )

        group_box = widgets.VBox(rows, layout=widgets.Layout(gap='8px', padding='8px 4px'))
        group_boxes.append(group_box)

    accordion = widgets.Accordion(
        children=group_boxes,
        layout=widgets.Layout(width='100%', max_height='560px', overflow_y='auto')
    )

    for i, group in enumerate(sorted_groups):
        accordion.set_title(i, f'{group} ({len(feature_groups[group])})')

    if sorted_groups:
        accordion.selected_index = 0

    group_selector = widgets.Dropdown(
        options=[(f'{g} ({len(feature_groups[g])})', i) for i, g in enumerate(sorted_groups)],
        description='Jump to:',
        style={'description_width': '80px'},
        layout=widgets.Layout(width='360px')
    )

    search_box = widgets.Text(
        value='',
        placeholder='Type feature name (e.g., proto, flow, dns) then press Enter',
        description='Find:',
        style={'description_width': '80px'},
        layout=widgets.Layout(width='620px')
    )

    predict_btn = widgets.Button(
        description='Predict',
        button_style='success',
        icon='check',
        layout=widgets.Layout(width='130px', height='36px')
    )
    reset_btn = widgets.Button(
        description='Reset',
        icon='refresh',
        layout=widgets.Layout(width='110px', height='36px')
    )

    result_out = widgets.Output()

    def on_group_change(change):
        if change['name'] == 'value':
            accordion.selected_index = change['new']

    def on_find_submit(_):
        q = search_box.value.strip().lower()
        if not q:
            return
        for idx, group in enumerate(sorted_groups):
            for col in feature_groups[group]:
                if q in col.lower():
                    accordion.selected_index = idx
                    with result_out:
                        clear_output()
                        print(f'Found in group: {group}. Scroll to locate "{col}".')
                    return
        with result_out:
            clear_output()
            print(f'No feature matched "{q}".')

    group_selector.observe(on_group_change, names='value')
    search_box.on_submit(on_find_submit)

    def _build_input_row():
        row = {}
        for c, d in CIC_DEFAULTS.items():
            v = input_widgets[c].value
            # All inputs are now assumed numerical as per user request.
            # Attempt conversion to int if the default was int, otherwise float.
            if isinstance(d, int) and not isinstance(d, bool):
                row[c] = int(v)
            else: # Default to float for all other cases
                row[c] = float(v)
        return row

    def _prepare_X(df_input):
        # Ensure df_input is a DataFrame before manipulation
        if not isinstance(df_input, pd.DataFrame):
            df_input = pd.DataFrame([df_input])

        # IMPORTANT: Align input features to what the model expects, dropping extra ones
        # and adding missing ones with fill_value (0.0)
        if INFER_FEATURE_NAMES is not None:
            df_input = df_input.reindex(columns=INFER_FEATURE_NAMES, fill_value=0.0)
            # Ensure all relevant columns are numeric after reindexing
            df_input = df_input.astype(float)
        else:
            # This case means feature_names could not be resolved, which should have
            # been caught earlier in the XAI setup cell. If not, proceed carefully.
            print("Warning: INFER_FEATURE_NAMES not resolved. Proceeding without explicit feature alignment.")


        X_proc = None
        if INFER_PREPROCESSOR is not None:
            # The df_input is now guaranteed to have the correct columns and order
            X_proc = INFER_PREPROCESSOR.transform(df_input)

        if X_proc is None: # This block is for when INFER_PREPROCESSOR is None or failed
            # If no preprocessor, or preprocessor didn't return X_proc,
            # directly use the aligned df_input values.
            X_proc = df_input.values

        # Final validation of feature count against the model's expectation
        expected_n = getattr(INFER_MODEL, 'n_features_in_', None)
        if expected_n is not None and X_proc.shape[1] != int(expected_n):
            raise ValueError(f'Feature count mismatch after preprocessing: expected {int(expected_n)}, got {X_proc.shape[1]}. '
                             'This indicates a fundamental mismatch between model and input definitions.')

        # If model expects named features (like some sklearn models or LightGBM),
        # ensure X_proc is a DataFrame with those names.
        if hasattr(INFER_MODEL, 'feature_names_in_') and isinstance(X_proc, np.ndarray):
            return pd.DataFrame(X_proc, columns=[str(c) for c in INFER_MODEL.feature_names_in_])
        return X_proc

    def on_predict(_):
        with result_out:
            clear_output()
            try:
                row = _build_input_row()
                # df_input = pd.DataFrame([row]) # df_input will be handled by _prepare_X
                X_input = _prepare_X(row)

                pred = int(INFER_MODEL.predict(X_input)[0])
                label = 'ATTACK' if pred == 1 else 'NORMAL'
                color = '#d32f2f' if pred == 1 else '#388e3c'

                score_html = ''
                if hasattr(INFER_MODEL, 'predict_proba'):
                    p = INFER_MODEL.predict_proba(X_input)[0]
                    normal_score = float(p[0])
                    attack_score = float(p[1])
                    pred_score = attack_score if pred == 1 else normal_score
                    score_html = (
                        f"<div style='margin-top:8px;font-size:14px;color:#444;'>"
                        f"normal_score={normal_score:.4f} | attack_score={attack_score:.4f} | "
                        f"prediction_confidence={pred_score:.4f}</div>"
                    )
                elif hasattr(INFER_MODEL, 'decision_function'):
                    decision_score = float(np.ravel(INFER_MODEL.decision_function(X_input))[0])
                    score_html = f"<div style='margin-top:8px;font-size:14px;color:#444;'>decision_score={decision_score:.6f}</div>"

                display(HTML(f"""
                    <div style='border:2px solid {color}; border-radius:8px; padding:14px; max-width:700px;'>
                        <div style='font-size:22px; font-weight:700; color:{color};'>Prediction: {label}</div>
                        <div style='margin-top:6px; font-size:13px; color:#666;'>Dataset: CICIoT | Model: DecisionTree</div>
                        {score_html}
                    </div>
                """))
            except Exception as e:
                print(f'Prediction failed: {e}')

    def on_reset(_):
        for c, d in CIC_DEFAULTS.items():
            input_widgets[c].value = d
        with result_out:
            clear_output()
            print('Inputs reset to CICIoT feature defaults.')

    predict_btn.on_click(on_predict)
    reset_btn.on_click(on_reset)

    controls = widgets.HBox(
        [group_selector, predict_btn, reset_btn],
        layout=widgets.Layout(gap='10px', align_items='center')
    )

    display(widgets.VBox([
        widgets.HTML('<h3 style="margin:0;">CIC-IoT Interactive Inference</h3>'),
        widgets.HTML('<div style="color:#555;">Edit values below and click Predict to classify as NORMAL or ATTACK.</div>'),
        search_box,
        controls,
        accordion,
        result_out
    ], layout=widgets.Layout(gap='10px', width='100%')))

## Anomaly Analysis Summary
Rule-of-thumb analysis for anomaly models (without SHAP/LIME):
- Use `decision_function` and `score_samples` for anomaly confidence.
- Lower decision score generally means more anomalous behavior.
- For LOF, neighborhood density differences drive outlier decisions.

In [ ]:
# Diagnostics + diagrams for anomaly score behavior
if 'INFER_MODEL' not in globals():
    print('Run setup cell first.')
else:
    import matplotlib.pyplot as plt

    split_dir = BASE_DIR / 'splits' / 'CICIoT2023'
    x_test_proc_path = split_dir / 'X_test_proc.npy'

    X_eval = None # Initialize X_eval

    if x_test_proc_path.exists():
        X_eval = np.load(x_test_proc_path)
    elif 'X_test' in globals() and INFER_PREPROCESSOR is not None:
        # If preprocessed X_test is not found, use the raw X_test and preprocess it.
        # Ensure it's in a DataFrame format expected by the preprocessor if INFER_FEATURE_NAMES are available.
        if INFER_FEATURE_NAMES is not None:
            X_eval_df = pd.DataFrame(X_test, columns=INFER_FEATURE_NAMES)
        else:
            X_eval_df = pd.DataFrame(X_test) # Fallback if feature names are not available

        X_eval = INFER_PREPROCESSOR.transform(X_eval_df) # Assuming preprocessor expects DataFrame or array
        X_eval = np.asarray(X_eval) # Ensure it's a numpy array

    if X_eval is None:
        print('No evaluation matrix found for diagnostics.')
    else:
        n = min(5000, X_eval.shape[0])
        X_sub = X_eval[:n]
        preds = INFER_MODEL.predict(X_sub)
        outlier_rate = float((preds == -1).mean())
        inlier_rate = 1.0 - outlier_rate

        print(f'Samples evaluated: {n}')
        print(f'Outlier rate: {outlier_rate:.4f}')

        # Diagram 1: Inlier vs Outlier rate
        plt.figure(figsize=(5, 4))
        plt.bar(['Inlier (+1)', 'Outlier (-1)'], [inlier_rate, outlier_rate], color=['#388e3c', '#d32f2f'])
        plt.ylim(0, 1)
        plt.title('Anomaly Label Distribution')
        plt.ylabel('Rate')
        plt.tight_layout()
        plt.show()

        # Diagram 2: Decision score distribution (when available)
        if hasattr(INFER_MODEL, 'decision_function'):
            d = np.ravel(INFER_MODEL.decision_function(X_sub))
            print(f'decision_score mean={float(d.mean()):.6f}, std={float(d.std()):.6f}, min={float(d.min()):.6f}, max={float(d.max()):.6f}')

            plt.figure(figsize=(8, 4.5))
            plt.hist(d, bins=40, color='#1976d2', alpha=0.85)
            plt.axvline(0.0, color='black', linestyle='--', linewidth=1.2, label='0 threshold')
            plt.title('Decision Score Distribution')
            plt.xlabel('decision_function score (lower => more anomalous)')
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

        # Diagram 3: score_samples distribution (when available)
        if hasattr(INFER_MODEL, 'score_samples'):
            s = np.ravel(INFER_MODEL.score_samples(X_sub))
            plt.figure(figsize=(8, 4.5))
            plt.hist(s, bins=40, color='#6a1b9a', alpha=0.85)
            plt.title('Sample Score Distribution')
            plt.xlabel('score_samples value')
            plt.ylabel('Count')
            plt.tight_layout()
            plt.show()